# hparam-precedence-merge — ex2: filter sentinel None from CLI before merging

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `hparam-precedence-merge`. Running the final beacon cell reports progress against the `Config: hparam precedence merge` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: hparam precedence merge` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`hparam-precedence-merge`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "hparam-precedence-merge"
DD_SUBTOPIC = "Config: hparam precedence merge"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Precedence merge — filter sentinel `None` from CLI

Ex1 implemented the strict `defaults < file < cli` chain — every key in `cli_args` wins, including explicit `None`. That works when argparse populates `cli_args` only for flags the user actually typed.

Many real CLIs (esp. fire / typer / hand-rolled argparse) emit `None` as a sentinel meaning 'flag not provided'. In that case you DO want to drop Nones from `cli_args` BEFORE merging, so an un-passed `--lr` doesn't blast away a value the YAML file just set:

```python
def merge_with_cli_sentinel(defaults, file_cfg, cli_args):
    cli_filtered = {k: v for k, v in cli_args.items() if v is not None}
    out = {}
    out.update(defaults)
    out.update(file_cfg)
    out.update(cli_filtered)
    return out
```

**Two different defaults — pick by convention.** Strict later-wins (ex1) is right when None is a legitimate value. Sentinel-filter (ex2) is right when None means 'unset'. The decision belongs at the CLI-parser layer, not buried in the merge function — but the helper must MATCH the parser's convention or you get silent precedence bugs.

**Why filter only the CLI layer, not the file.** A YAML user who types `lr: null` is making an explicit choice; the file is expressive enough to mean what it says. The CLI layer has the shape problem because argparse defaults to None for absent flags.

### Exercise 2 — filter sentinel None from CLI before merging

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a sentinel-None dict-comprehension filter to `cli_args` BEFORE chained `dict.update` so that argparse-style 'flag not provided' Nones don't overwrite values set by the YAML config layer.
> Keywords: config, precedence, sentinel-none, argparse-default
> ```

**KCs targeted:** `dict-comprehension-filter-none`, `dict-update-later-wins`

Implement `ex2_merge_filter_cli(defaults, file_cfg, cli_args)`.

Same three-layer chain as ex1, but with a twist: in this merge, `cli_args` came from argparse where ABSENT flags default to `None`. We don't want those Nones blasting away values set in the file layer.

Algorithm:
1. Build `cli_filtered = {k: v for k, v in cli_args.items()   if v is not None}`.
2. `out = {}`; `out.update(defaults)`; `out.update(file_cfg)`;   `out.update(cli_filtered)`.
3. Return `out`.

Constraints:
- No mutation of any input dict (including `cli_args`).
- Non-None CLI values STILL override file (later-wins).
- A file value of `None` should pass through (only CLI is   filtered).
- A file value that's overridden by a non-None CLI value   must be replaced.
- A CLI value of `None` must NOT override the file.

In [ ]:
def ex2_merge_filter_cli(defaults, file_cfg, cli_args):
    cli_filtered = {k: v for k, v in cli_args.items() if v is not None}
    out = {}
    out.update(defaults)
    out.update(file_cfg)
    out.update(cli_filtered)
    return out


<details><summary>Solution</summary>

```python
def ex2_merge_filter_cli(defaults, file_cfg, cli_args):
    cli_filtered = {k: v for k, v in cli_args.items() if v is not None}
    out = {}
    out.update(defaults)
    out.update(file_cfg)
    out.update(cli_filtered)
    return out
```

**Why filter only the CLI layer.** A YAML config value of `None` is an explicit user choice — the file format is expressive enough to mean what it says. The CLI layer has the sentinel-None problem because argparse defaults absent flags to None.

**Comprehension vs `{**d}` spread.** `{k: v for k, v in cli.items() if v is not None}` is the clearest filter. `{k: v for k, v in cli.items() if v != None}` would catch numpy/pandas NaT-style objects too — usually not what you want.

**Choosing between ex1 (strict) and ex2 (sentinel) behavior.** Match your CLI parser. argparse with default=None -> ex2. Hand-rolled parsers that only include user-typed flags -> ex1. Picking the wrong one is one of the most common silent precedence bugs in training scripts.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()